# ChroniclingAmericaQA — Phi-3.5 Zero-Shot Baseline

**Dataset:** [Bhawna/ChroniclingAmericaQA](https://huggingface.co/datasets/Bhawna/ChroniclingAmericaQA)  
**Model:** `microsoft/phi-3.5-mini-instruct`  
**Method:** Zero-shot instruction prompting → baseline metrics for QLoRA fine-tuning

This notebook covers:
1. Setup and imports
2. Configuration
3. Load dataset
4. Exploratory data inspection
5. Preprocessing (whitespace normalization, truncation, OCR toggle)
6. Prompt formatting (Phi-3.5 chat template)
7. Zero-shot baseline inference
8. Baseline evaluation (EM, Token F1, ROUGE-L)

---
## 1. Setup and Imports

In [3]:
# Uncomment and run once to install / pin dependencies
!pip install "transformers==4.44.2" datasets peft trl bitsandbytes accelerate
!pip install evaluate rouge_score sentencepiece
!pip install pandas numpy tqdm matplotlib

In [4]:
import os
import re
import sys
import json
import string
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from datasets import load_dataset
from transformers import set_seed
import evaluate as hf_evaluate

warnings.filterwarnings("ignore")
print("Imports OK.")

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


Imports OK.


---
## 2. Configuration

All hyperparameters live here. Adjust before running any downstream cell.

In [20]:
from google.colab import drive
drive.mount('/content/drive')

# ── Dataset ─────────────────────────────────────────────────────────────────
DATASET_NAME = "Bhawna/ChroniclingAmericaQA"

MAX_TRAIN_SAMPLES = 500    # None = full split
MAX_DEV_SAMPLES   = 100
MAX_TEST_SAMPLES  = 200

USE_RAW_OCR = False        # True = use raw OCR instead of cleaned context

# ── Sequence lengths ─────────────────────────────────────────────────────────
MAX_INPUT_LENGTH  = 1024   # approximate token budget for context + prompt
MAX_TARGET_LENGTH = 64     # max answer length in tokens

# ── Model ────────────────────────────────────────────────────────────────────
MODEL_NAME          = "microsoft/phi-3.5-mini-instruct"
BASELINE_BATCH_SIZE = 4    # reduce if OOM
GEN_MAX_NEW_TOKENS  = 64
GEN_DO_SAMPLE       = False  # greedy decoding for reproducibility

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED       = 42
OUTPUT_DIR = Path("/content/drive/MyDrive/chronicling_qa")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
print("Configuration loaded.")
print(f"  Dataset    : {DATASET_NAME}")
print(f"  Model      : {MODEL_NAME}")
print(f"  USE_RAW_OCR: {USE_RAW_OCR}")
print(f"  Output dir : {OUTPUT_DIR.resolve()}")

Mounted at /content/drive
Configuration loaded.
  Dataset    : Bhawna/ChroniclingAmericaQA
  Model      : microsoft/phi-3.5-mini-instruct
  USE_RAW_OCR: False
  Output dir : /content/drive/MyDrive/chronicling_qa


---
## 3. Load Dataset

Load ChroniclingAmericaQA from the Hugging Face Hub and define field name constants.

In [6]:
# from huggingface_hub import login; login(token="hf_YOUR_TOKEN_HERE")

print(f"Loading dataset: {DATASET_NAME} ...")
raw_datasets = load_dataset(DATASET_NAME)

train_ds = raw_datasets["train"]
dev_ds   = raw_datasets["validation"]
test_ds  = raw_datasets["test"]

# Field name constants
FIELD_QUESTION = "question"
FIELD_ANSWER   = "answer"
FIELD_CONTEXT  = "context"   # cleaned / gold context
FIELD_RAW_OCR  = "raw_ocr"   # raw OCR text
HAS_RAW_OCR    = FIELD_RAW_OCR in train_ds.column_names

print(f"\nSplits  — Train: {len(train_ds):,}  Dev: {len(dev_ds):,}  Test: {len(test_ds):,}")
print(f"Columns : {train_ds.column_names}")
print(f"HAS_RAW_OCR: {HAS_RAW_OCR}")

Loading dataset: Bhawna/ChroniclingAmericaQA ...

Splits  — Train: 439,302  Dev: 24,111  Test: 24,084
Columns : ['query_id', 'question', 'answer', 'org_answer', 'para_id', 'context', 'raw_ocr', 'publication_date', 'trans_que', 'trans_ans', 'url']
HAS_RAW_OCR: True


---
## 4. Exploratory Data Inspection

Understand the data schema, typical answer lengths, and OCR noise.

In [7]:
for i, example in enumerate(train_ds.select(range(3))):
    print(f"{'='*70}")
    print(f"Example {i+1}")
    print(f"{'='*70}")
    print(f"QUESTION : {example['question']}")
    print(f"ANSWER   : {example['answer']}")
    print(f"CONTEXT  : {example['context'][:300]}...")
    if HAS_RAW_OCR:
        print(f"RAW OCR  : {example['raw_ocr'][:300]}...")
    print()

Example 1
QUESTION : Who is the author of the book, "Horrors of Slavery, or the American Turf in Tripoli"?
ANSWER   : WILLIAM RAY
CONTEXT  : Aiscellaneous Repository. From the Albany Register, WAR, OR A PROSPECT OF IT, From recent instances of British Outrage. BY: WILLIAM RAY, Author of the contemplated publication, entitled, “Horrors of Slavery, or the American Turf in Tripoli,” VOTARIES of Freedom, arm! The British Lion roars! Legions ...
RAW OCR  : fAiscellancous Bepogitory.
. dvom the Albany Regifier,
. . WAR,OR A PROSPECT OF IT,
From recent inflances of Britifp Oulrage.
 BY: WILLIAM RAY:
HAuthsr of the tontemplated publication, entitled,
« Horrors of Slavery,or the American Turg
in Tripoli,”
VOT’RIES of Freedom, arm!
 The British Lion roars ...

Example 2
QUESTION : Who was the Grand Officer of the Legion of Honor?
ANSWER   : de Rosemberg
CONTEXT  : Surely he above the rest of his fellow mortals, partakes of heaven here below, of bliss which none but the virtuous ever claim. ¥ Obi

In [8]:
def compute_length_stats(dataset, field, label):
    lengths = [len(str(ex.get(field, "")).split()) for ex in dataset]
    return {
        "field": label,
        "mean":   round(float(np.mean(lengths)), 1),
        "median": round(float(np.median(lengths)), 1),
        "p95":    round(float(np.percentile(lengths, 95)), 1),
        "max":    int(np.max(lengths)),
    }

stats_rows = [
    compute_length_stats(train_ds, FIELD_QUESTION, "Question"),
    compute_length_stats(train_ds, FIELD_ANSWER,   "Answer"),
    compute_length_stats(train_ds, FIELD_CONTEXT,  "Context (clean)"),
]
if HAS_RAW_OCR:
    stats_rows.append(compute_length_stats(train_ds, FIELD_RAW_OCR, "Context (OCR)"))

stats_df = pd.DataFrame(stats_rows).set_index("field")
print("Word-count statistics (train split):")
display(stats_df)

Word-count statistics (train split):


,mean,median,p95,max
field,,,,
Question,11.1,10.0,18.0,30
Answer,2.0,2.0,4.0,47
Context (clean),219.7,225.0,245.0,4077
Context (OCR),233.9,240.0,250.0,1115


---
## 5. Preprocessing

Minimal, reversible preprocessing: choose OCR vs. clean context, normalize
whitespace, handle missing values, and truncate long passages.

In [9]:
CONTEXT_MAX_WORDS = int(MAX_INPUT_LENGTH * 0.6)


def normalize_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def get_context(example: dict, use_raw_ocr: bool = False) -> str:
    field = FIELD_RAW_OCR if (use_raw_ocr and HAS_RAW_OCR) else FIELD_CONTEXT
    return normalize_whitespace(str(example.get(field, "") or ""))


def preprocess_example(example: dict) -> dict:
    question = normalize_whitespace(str(example.get(FIELD_QUESTION, "") or ""))
    answer   = normalize_whitespace(str(example.get(FIELD_ANSWER,   "") or ""))
    context  = get_context(example, use_raw_ocr=USE_RAW_OCR)
    context  = " ".join(context.split()[:CONTEXT_MAX_WORDS])
    return {"question": question, "answer": answer, "context": context}


print("Applying preprocessing ...")
train_clean = train_ds.map(preprocess_example, desc="Preprocess train")
dev_clean   = dev_ds.map(preprocess_example,   desc="Preprocess dev")
test_clean  = test_ds.map(preprocess_example,  desc="Preprocess test")

ex = train_clean[0]
assert ex["question"], "Question should not be empty after preprocessing"
print(f"\nSample:")
print(f"  Q: {ex['question']}")
print(f"  A: {ex['answer']}")
print(f"  Context ({len(ex['context'].split())} words): {ex['context'][:200]}...")

Applying preprocessing ...

Sample:
  Q: Who is the author of the book, "Horrors of Slavery, or the American Turf in Tripoli"?
  A: WILLIAM RAY
  Context (196 words): Aiscellaneous Repository. From the Albany Register, WAR, OR A PROSPECT OF IT, From recent instances of British Outrage. BY: WILLIAM RAY, Author of the contemplated publication, entitled, “Horrors of S...


---
## 6. Prompt Formatting

Instruction-style prompt compatible with Phi-3.5's chat template.
The same helpers are shared between inference and training.

In [10]:
SYSTEM_PROMPT = (
    "You are answering questions about historical American newspapers. "
    "Use the provided context to answer the question briefly and accurately. "
    'If the answer is not supported by the context, say "Not enough information.".'
)


def format_prompt_for_inference(context: str, question: str) -> str:
    user_msg = f"Context:\n{context}\n\nQuestion:\n{question}\n\nAnswer:"
    return (
        f"<|system|>\n{SYSTEM_PROMPT}<|end|>\n"
        f"<|user|>\n{user_msg}<|end|>\n"
        "<|assistant|>\n"
    )


def format_prompt_for_training(context: str, question: str, answer: str) -> str:
    return format_prompt_for_inference(context, question) + answer + "<|end|>"


# Sanity check
sample = train_clean[0]
print("=== Inference prompt (first 500 chars) ===")
print(format_prompt_for_inference(sample["context"], sample["question"])[:500])
print("\n=== Training sequence tail ===")
full = format_prompt_for_training(sample["context"], sample["question"], sample["answer"])
print(full[-80:])

=== Inference prompt (first 500 chars) ===
<|system|>
You are answering questions about historical American newspapers. Use the provided context to answer the question briefly and accurately. If the answer is not supported by the context, say "Not enough information.".<|end|>
<|user|>
Context:
Aiscellaneous Repository. From the Albany Register, WAR, OR A PROSPECT OF IT, From recent instances of British Outrage. BY: WILLIAM RAY, Author of the contemplated publication, entitled, “Horrors of Slavery, or the American Turf in Tripoli,” VOTARI

=== Training sequence tail ===
 the American Turf in Tripoli"?

Answer:<|end|>
<|assistant|>
WILLIAM RAY<|end|>


---
## 7. Zero-Shot Baseline Inference

Load `microsoft/phi-3.5-mini-instruct` without fine-tuning and run greedy
generation on the dev subset to establish a pre-training baseline.

> **Note:** requires `transformers==4.44.2` — the model's bundled
> `modeling_phi3.py` uses `DynamicCache.from_legacy_cache` which was removed
> in later versions. The cell below flushes any cached newer version.

In [11]:
# Ensure transformers 4.44.2 is active (compatible with phi-3.5 custom modeling)
import importlib
for mod in [k for k in sys.modules if k.startswith("transformers")]:
    del sys.modules[mod]

from transformers import AutoTokenizer, AutoModelForCausalLM
import transformers
print(f"transformers: {transformers.__version__}  |  CUDA: {torch.cuda.is_available()}")

baseline_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
baseline_tokenizer.pad_token    = baseline_tokenizer.eos_token
baseline_tokenizer.padding_side = "left"   # required for left-padded batch generation

baseline_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
baseline_model.eval()
print(f"Model on : {next(baseline_model.parameters()).device}")
print(f"Params   : {sum(p.numel() for p in baseline_model.parameters()):,}")

transformers: 4.44.2  |  CUDA: False


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model on : cpu
Params   : 3,821,079,552


In [22]:
import shutil
from pathlib import Path
import pandas as pd

def generate_answers_batch(examples, tokenizer, model, batch_size=BASELINE_BATCH_SIZE):
    """Batched greedy generation; returns one answer string per example."""
    all_preds = []
    n = len(examples)
    for start in tqdm(range(0, n, batch_size), desc="Inference"):
        # Use .select() — Dataset[a:b] returns dict-of-lists, not list-of-dicts
        batch   = examples.select(range(start, min(start + batch_size, n)))
        prompts = [
            format_prompt_for_inference(batch[i]["context"], batch[i]["question"])
            for i in range(len(batch))
        ]
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding="max_length", # padded for TPU optimization
            truncation=True,
            max_length=MAX_INPUT_LENGTH,
        ).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=GEN_MAX_NEW_TOKENS,
                do_sample=GEN_DO_SAMPLE,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        for i, out in enumerate(output_ids):
            new_tokens = out[inputs["input_ids"].shape[1]:]
            decoded    = tokenizer.decode(new_tokens, skip_special_tokens=True)
            all_preds.append(decoded.strip().split("\n")[0].strip())

    return all_preds


# ── Run inference & Caching ──────────────────────────────────────────────────
baseline_csv = OUTPUT_DIR / "dev_predictions_baseline.csv"
local_baseline_csv = Path("/content/outputs/chronicling_qa/dev_predictions_baseline.csv")

# Migrate old local file to Drive if it exists
if local_baseline_csv.exists() and not baseline_csv.exists():
    print(f"Migrating local predictions to Drive: {baseline_csv}")
    shutil.copy(local_baseline_csv, baseline_csv)

if baseline_csv.exists():
    print(f"Loading cached baseline predictions from {baseline_csv} ...")
    baseline_df = pd.read_csv(baseline_csv)
    baseline_preds = baseline_df["prediction"].tolist()
    baseline_golds = baseline_df["gold_answer"].tolist()
    baseline_questions = baseline_df["question"].tolist()
else:
    print("No cache found. Running inference...")
    dev_eval           = dev_clean.select(range(min(MAX_DEV_SAMPLES, len(dev_clean))))
    baseline_preds     = generate_answers_batch(dev_eval, baseline_tokenizer, baseline_model)
    baseline_golds     = [dev_eval[i]["answer"]   for i in range(len(dev_eval))]
    baseline_questions = [dev_eval[i]["question"] for i in range(len(dev_eval))]

    # ── Save to CSV ───────────────────────────────────────────────────────────
    baseline_df  = pd.DataFrame({
        "question":    baseline_questions,
        "gold_answer": baseline_golds,
        "prediction":  baseline_preds,
    })
    baseline_df.to_csv(baseline_csv, index=False)
    print(f"Saved {len(baseline_df)} predictions → {baseline_csv}")

print(f"\nFirst 5 predictions vs. gold:")
for q, pred, gold in zip(baseline_questions[:5], baseline_preds[:5], baseline_golds[:5]):
    print(f"  Q   : {q}")
    print(f"  Pred: {pred}")
    print(f"  Gold: {gold}\n")

Migrating local predictions to Drive: /content/drive/MyDrive/chronicling_qa/dev_predictions_baseline.csv
Loading cached baseline predictions from /content/drive/MyDrive/chronicling_qa/dev_predictions_baseline.csv ...

First 5 predictions vs. gold:
  Q   : How much of the crew would Gerry want to shore up in a gale of wind?
  Pred: Half of the crew
  Gold: half

  Q   : How many Irish Linens are low-priced?
  Pred: 30 Dozen Irish Linens are low-priced.
  Gold: 300 Dozen

  Q   : How long does it take for the heirs of the said Israel to appear before the Judge?
  Pred: The context does not provide a specific time frame for the heirs to appear before the Judge. It only states that they are to appear on the 15th day of June. Therefore, there is not enough information to determine the exact time it takes for them to appear.
  Gold: three weeks

  Q   : Whose Store did JOHN DORRANCE receive the Reports of the Case?
  Pred: W.R. Wilpre's Store
  Gold: Allspice

  Q   : How many Delaware count

---
## 8. Baseline Evaluation

Compute **Exact Match (EM)**, **Token F1** (SQuAD-style), and **ROUGE-L**
for the zero-shot baseline predictions.

In [14]:
# ── Normalization (SQuAD-style) ──────────────────────────────────────────────

def normalize_answer(s: str) -> str:
    """Lowercase, strip punctuation, remove articles, collapse whitespace."""
    s = s.lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = "".join(ch for ch in s if ch not in string.punctuation)
    return " ".join(s.split())


def get_tokens(s: str) -> list:
    return normalize_answer(s).split()


# ── Exact Match ───────────────────────────────────────────────────────────────

def exact_match(prediction: str, gold: str) -> int:
    return int(normalize_answer(prediction) == normalize_answer(gold))


# ── Token F1 ──────────────────────────────────────────────────────────────────

def token_f1(prediction: str, gold: str) -> float:
    pred_tokens = get_tokens(prediction)
    gold_tokens = get_tokens(gold)
    common   = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall    = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


# ── ROUGE-L ───────────────────────────────────────────────────────────────────

rouge_metric = hf_evaluate.load("rouge")

def compute_rouge_l(predictions: list, references: list) -> float:
    return rouge_metric.compute(
        predictions=predictions,
        references=references,
        rouge_types=["rougeL"],
        use_stemmer=True,
    )["rougeL"]


# ── Aggregate ─────────────────────────────────────────────────────────────────

def compute_metrics(predictions: list, references: list, label: str = "") -> dict:
    em_scores = [exact_match(p, g) for p, g in zip(predictions, references)]
    f1_scores = [token_f1(p, g)    for p, g in zip(predictions, references)]
    return {
        "split":   label,
        "n":       len(predictions),
        "EM":      round(100 * sum(em_scores) / len(em_scores), 2),
        "F1":      round(100 * sum(f1_scores) / len(f1_scores), 2),
        "ROUGE-L": round(100 * compute_rouge_l(predictions, references), 2),
    }


# ── Results ───────────────────────────────────────────────────────────────────

baseline_metrics = compute_metrics(
    baseline_preds, baseline_golds, label="baseline (zero-shot)"
)

print("=" * 50)
print(f"  Baseline Zero-Shot  (n={baseline_metrics['n']})")
print("=" * 50)
print(f"  Exact Match : {baseline_metrics['EM']:.2f}%")
print(f"  Token F1    : {baseline_metrics['F1']:.2f}%")
print(f"  ROUGE-L     : {baseline_metrics['ROUGE-L']:.2f}%")
print("=" * 50)

  Baseline Zero-Shot  (n=100)
  Exact Match : 13.00%
  Token F1    : 31.34%
  ROUGE-L     : 32.44%


---
## Section 9 — Training Configuration

All training hyperparameters live here. `TRAINING_MODE` switches between
LoRA and full fine-tuning. LoRA uses `phi-3.5-mini-instruct` in **fp16**
(no 4-bit quantisation so results are unconfounded). Full fine-tuning uses
`phi-2` (2.7 B) for compute feasibility.

In [21]:
from pathlib import Path
import torch
from transformers import set_seed

# ── Models ────────────────────────────────────────────────────────────────────
MODEL_NAME_LORA    = "microsoft/phi-3.5-mini-instruct"

# ── Sample limits ─────────────────────────────────────────────────────────────
MAX_TRAIN_SAMPLES_FT = 500       # set None for full dataset (very slow)
MAX_DEV_SAMPLES_FT   = 100
MAX_TEST_SAMPLES_FT  = 200

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE        = 2
GRAD_ACCUM_STEPS  = 8            # effective batch size = 16
NUM_EPOCHS        = 3
LEARNING_RATE     = 2e-4
MAX_SEQ_LEN       = 768          # max tokens for training sequences

# ── LoRA ──────────────────────────────────────────────────────────────────────
LORA_RANK         = 16
LORA_ALPHA        = 32
LORA_DROPOUT      = 0.05
LORA_TARGET_MODULES = [
    "q_proj", "v_proj", "k_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

LORA_RANKS = [8, 16, 32, 64]    # rank ablation sweep

# ── Paths ─────────────────────────────────────────────────────────────────────
FT_OUTPUT_DIR = Path("/content/drive/MyDrive/chronicling_qa/fine_tuned")
FT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Precision ─────────────────────────────────────────────────────────────────
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16

set_seed(SEED)   # SEED defined in original config cell

print(f"DEVICE        : {DEVICE}")
print(f"precision     : {'bf16' if USE_BF16 else 'fp16' if USE_FP16 else 'fp32'}")
print(f"LORA_RANK     : {LORA_RANK}")
print(f"NUM_EPOCHS    : {NUM_EPOCHS}")
print(f"FT_OUTPUT_DIR : {FT_OUTPUT_DIR.resolve()}")

DEVICE        : cpu
precision     : fp32
LORA_RANK     : 16
NUM_EPOCHS    : 3
FT_OUTPUT_DIR : /content/drive/MyDrive/chronicling_qa/fine_tuned


---
## Section 10 — Temporal Partitioning

We split the dataset into three equal-sized temporal bins (**early / mid /
late**) derived from the `publication_date` field. Bin boundaries are
computed as tertiles from the *training* distribution so the split is
data-driven and reproducible.

In [23]:
import re, json as _json
import numpy as np
from collections import Counter as _Counter
from datasets import Dataset, load_from_disk
from pathlib import Path

TIME_BIN_CACHE = OUTPUT_DIR / "time_binned"
TIME_BIN_META  = OUTPUT_DIR / "time_bin_meta.json"


def extract_year(example):
    raw = str(example.get("publication_date", "") or "")
    m = re.search(r"(1[6-9]\d{2}|20[01]\d)", raw)
    return int(m.group(1)) if m else -1


def assign_time_bin(year):
    if year <= 0:
        return "unknown"
    if year <= EARLY_MAX:
        return "early"
    if year <= MID_MAX:
        return "mid"
    return "late"


def add_time_bin(dataset):
    def _tag(ex):
        ex["time_bin"] = assign_time_bin(extract_year(ex))
        return ex
    return dataset.map(_tag, desc="Assign time bin")


# ── Load from cache or recompute ─────────────────────────────────────────────
if TIME_BIN_CACHE.exists() and TIME_BIN_META.exists():
    print("Loading binned datasets from cache ...")
    meta = _json.loads(TIME_BIN_META.read_text())
    EARLY_MAX = meta["EARLY_MAX"]
    MID_MAX   = meta["MID_MAX"]
    train_binned = load_from_disk(str(TIME_BIN_CACHE / "train"))
    dev_binned   = load_from_disk(str(TIME_BIN_CACHE / "dev"))
    test_binned  = load_from_disk(str(TIME_BIN_CACHE / "test"))
    print(f"  Loaded. Boundaries: early<={EARLY_MAX}, mid<={MID_MAX}")
else:
    print("Computing temporal bin boundaries from training data ...")
    _train_years = [extract_year(ex) for ex in train_ds]
    _valid_years = [y for y in _train_years if y > 0]
    EARLY_MAX = int(np.percentile(_valid_years, 33))
    MID_MAX   = int(np.percentile(_valid_years, 66))

    print(f"Temporal bin boundaries (from training tertiles):")
    print(f"  early : year <= {EARLY_MAX}")
    print(f"  mid   : {EARLY_MAX} < year <= {MID_MAX}")
    print(f"  late  : year > {MID_MAX}")

    print("\nTagging splits with time_bin ...")
    train_binned = add_time_bin(train_clean)
    dev_binned   = add_time_bin(dev_clean)
    test_binned  = add_time_bin(test_clean)

    print("Saving binned datasets to cache ...")
    TIME_BIN_CACHE.mkdir(parents=True, exist_ok=True)
    train_binned.save_to_disk(str(TIME_BIN_CACHE / "train"))
    dev_binned.save_to_disk(  str(TIME_BIN_CACHE / "dev"))
    test_binned.save_to_disk( str(TIME_BIN_CACHE / "test"))
    TIME_BIN_META.write_text(_json.dumps({"EARLY_MAX": EARLY_MAX, "MID_MAX": MID_MAX}))
    print(f"  Saved → {TIME_BIN_CACHE}")

for split_name, ds in [("train", train_binned), ("dev", dev_binned), ("test", test_binned)]:
    counts = _Counter(ds["time_bin"])
    print(f"  {split_name}: {dict(sorted(counts.items()))}")

TIME_BINS = ["early", "mid", "late"]

Computing temporal bin boundaries from training data ...
Temporal bin boundaries (from training tertiles):
  early : year <= 1865
  mid   : 1865 < year <= 1890
  late  : year > 1890

Tagging splits with time_bin ...
Saving binned datasets to cache ...


Saving the dataset (0/3 shards):   0%|          | 0/439302 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/24111 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/24084 [00:00<?, ? examples/s]

  Saved → /content/drive/MyDrive/chronicling_qa/time_binned
  train: {'early': 146502, 'late': 139268, 'mid': 153532}
  dev: {'early': 11435, 'late': 6844, 'mid': 5832}
  test: {'early': 11522, 'late': 6730, 'mid': 5832}


---
## Section 11 — Training Data Format & Tokenisation

We reuse `format_prompt_for_inference` / `format_prompt_for_training` from
Section 6 of the baseline notebook. Answer tokens receive real labels while
prompt tokens are masked (`-100`) so loss is computed only on the answer span.

In [24]:
from transformers import AutoTokenizer


def make_tokenize_fn(tokenizer, max_length=MAX_SEQ_LEN):
    # Prompt tokens masked (-100); loss computed only on the answer span.
    def _tokenize(example):
        prompt   = format_prompt_for_inference(example["context"], example["question"])
        full_seq = format_prompt_for_training(
            example["context"], example["question"], example["answer"]
        )
        prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
        full_enc   = tokenizer(
            full_seq,
            add_special_tokens=False,
            max_length=max_length,
            truncation=True,
            padding=False,
        )
        input_ids = full_enc["input_ids"]
        n_prompt  = min(len(prompt_ids), len(input_ids))
        labels    = [-100] * n_prompt + input_ids[n_prompt:]
        return {
            "input_ids":      input_ids,
            "attention_mask": full_enc["attention_mask"],
            "labels":         labels[:len(input_ids)],
        }
    return _tokenize


def make_collator(tokenizer):
    from transformers import DataCollatorForSeq2Seq
    return DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        padding=True,
        pad_to_multiple_of=8,
        label_pad_token_id=-100,
    )


print("Training data helpers ready.")
print(f"  MAX_SEQ_LEN : {MAX_SEQ_LEN}")

Training data helpers ready.
  MAX_SEQ_LEN : 768
